# Prompt Engineering

**Live online course — instructor walkthrough notebook**

This notebook follows the course lecture notes. Each section has:
- **Lecture notes** (markdown) — what to explain on the slide/screen.
- **Demo code** — a `run_prompt_vN()` function plus an immediate `evaluator.run_evaluation(...)` call, so the class sees a real score every time the prompt changes.
- **🏫 During class** callouts — specific instructor actions, questions to ask, and variations to try.

The running example throughout the module is a **one-day meal planner for athletes**. We start with a weak prompt, apply one technique at a time, and re-run the eval after each technique so students watch the average score climb in real time.

---

## Agenda

0. Setup (environment + eval pipeline)
1. Prompt Engineering — the loop  *(baseline eval run)*
2. Being Clear and Direct  *(eval re-run)*
3. Being Specific (attributes + steps)  *(eval re-run)*
4. Structure with XML Tags  *(eval re-run)*
5. Providing Examples (one-shot / multi-shot)  *(eval re-run)*
6. Recap + practice exercises

## 0. Setup (do this before class starts)

1. Install dependencies:
   ```bash
   pip install anthropic python-dotenv
   ```
2. Create a file named `.env` in the same directory as this notebook containing:
   ```
   ANTHROPIC_API_KEY="sk-ant-...your-key..."
   ```
3. Add `.env` to `.gitignore` so it is never committed to version control.

> **🏫 During class:** This notebook assumes students already sat through *Intro to Claude API* and *Prompt Evaluation*. A quick refresher: *"Last week we wrote an eval so we can measure prompts. Today we use that same eval to grade every version of our prompt — one run after each technique, so the score is the proof."*

In [ ]:
# Install packages (uncomment if not already installed)
%pip install anthropic python-dotenv

The setup cell below does four things students should recognize from *Intro to Claude API*:

1. `load_dotenv()` reads the `.env` file next to the notebook so `os.getenv("ANTHROPIC_API_KEY")` returns the secret — the key never appears in the notebook.
2. `client = Anthropic()` creates the SDK client with no arguments; it picks up `ANTHROPIC_API_KEY` from the environment automatically.
3. `model = "claude-haiku-4-5"` — we deliberately pick the **smallest** model for this module. Prompt engineering wins are most visible on a weaker model; a stronger model papers over bad prompts.
4. The three `print(...)` lines are a sanity check. If `Key loaded:` prints `False`, the `.env` file isn't where it needs to be — fix that before running any API call.

In [14]:
from dotenv import load_dotenv
from anthropic import Anthropic
import anthropic
import os

load_dotenv()  # loads ANTHROPIC_API_KEY from .env

client = Anthropic()  # picks up ANTHROPIC_API_KEY automatically

# Prompt engineering gains are most visible on a smaller model.
# We'll deliberately use Haiku so weak prompts look weak and strong prompts look strong.
model = "claude-haiku-4-5"

print("SDK version:", anthropic.__version__)
print("Model:", model)
print("Key loaded:", bool(os.getenv("ANTHROPIC_API_KEY")))

SDK version: 0.96.0
Model: claude-haiku-4-5
Key loaded: True


### Shared `chat()` helper

Every `run_prompt_vN` function in this notebook reuses the same `chat()` helper. After this cell runs, students should see we're only varying **one thing** per section — the prompt body itself.

In [15]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})
    return messages

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})
    return messages

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    """Send messages to Claude and return the assistant text."""
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    if system is not None:
        params["system"] = system
    if stop_sequences is not None:
        params["stop_sequences"] = stop_sequences
    response = client.messages.create(**params)
    return response.content[0].text

### Eval pipeline (same code as the prompt-evaluation module)

The next few cells paste in the eval pipeline we built last week — imports, the HTML report builder, and the `PromptEvaluator` class. We set this up **once, here**, then every section below calls `evaluator.run_evaluation(...)` against the same dataset. That way the only variable between sections is the prompt.

You don't need to read through the class in class; students already saw it. Tell them: *"This is the same PromptEvaluator from last week. Every section is going to call `run_evaluation` on a different prompt version and print the average score. Watch that number."*

In [16]:
import json
import concurrent.futures
import re
from textwrap import dedent
from statistics import mean

#### HTML report builder

Writes an `output_<version>.html` per eval run containing every test case, its generated output, the grader's score, and the reasoning. Open one in a browser during class so the room can see *why* a given version scored the way it did — the individual critiques are often more instructive than the average.

In [17]:
def generate_prompt_evaluation_report(evaluation_results):
    total_tests = len(evaluation_results)
    scores = [result["score"] for result in evaluation_results]
    avg_score = mean(scores) if scores else 0
    max_possible_score = 10
    pass_rate = (
        100 * len([s for s in scores if s >= 7]) / total_tests if total_tests else 0
    )

    html = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Prompt Evaluation Report</title>
        <style>
            body {{ font-family: Arial, sans-serif; line-height: 1.6; margin: 0; padding: 20px; color: #333; }}
            .header {{ background-color: #f0f0f0; padding: 20px; border-radius: 5px; margin-bottom: 20px; }}
            .summary-stats {{ display: flex; justify-content: space-between; flex-wrap: wrap; gap: 10px; }}
            .stat-box {{ background-color: #fff; border-radius: 5px; padding: 15px; box-shadow: 0 2px 5px rgba(0,0,0,0.1); flex-basis: 30%; min-width: 200px; }}
            .stat-value {{ font-size: 24px; font-weight: bold; margin-top: 5px; }}
            table {{ width: 100%; border-collapse: collapse; margin-top: 20px; }}
            th {{ background-color: #4a4a4a; color: white; text-align: left; padding: 12px; }}
            td {{ padding: 10px; border-bottom: 1px solid #ddd; vertical-align: top; width: 20%; }}
            tr:nth-child(even) {{ background-color: #f9f9f9; }}
            .score {{ font-weight: bold; padding: 5px 10px; border-radius: 3px; display: inline-block; }}
            .score-high {{ background-color: #c8e6c9; color: #2e7d32; }}
            .score-medium {{ background-color: #fff9c4; color: #f57f17; }}
            .score-low {{ background-color: #ffcdd2; color: #c62828; }}
            .output pre {{ background-color: #f5f5f5; border: 1px solid #ddd; border-radius: 4px; padding: 10px; margin: 0; font-family: monospace; font-size: 14px; white-space: pre-wrap; word-wrap: break-word; }}
            .score-col {{ width: 80px; }}
        </style>
    </head>
    <body>
        <div class="header">
            <h1>Prompt Evaluation Report</h1>
            <div class="summary-stats">
                <div class="stat-box"><div>Total Test Cases</div><div class="stat-value">{total_tests}</div></div>
                <div class="stat-box"><div>Average Score</div><div class="stat-value">{avg_score:.1f} / {max_possible_score}</div></div>
                <div class="stat-box"><div>Pass Rate (≥7)</div><div class="stat-value">{pass_rate:.1f}%</div></div>
            </div>
        </div>
        <table>
            <thead>
                <tr><th>Scenario</th><th>Prompt Inputs</th><th>Solution Criteria</th><th>Output</th><th>Score</th><th>Reasoning</th></tr>
            </thead>
            <tbody>
    """

    for result in evaluation_results:
        prompt_inputs_html = "<br>".join(
            [f"<strong>{k}:</strong> {v}" for k, v in result["test_case"]["prompt_inputs"].items()]
        )
        criteria_string = "<br>• ".join(result["test_case"]["solution_criteria"])
        score = result["score"]
        score_class = "score-high" if score >= 8 else "score-low" if score <= 5 else "score-medium"

        html += f"""
            <tr>
                <td>{result["test_case"]["scenario"]}</td>
                <td>{prompt_inputs_html}</td>
                <td>• {criteria_string}</td>
                <td class="output"><pre>{result["output"]}</pre></td>
                <td class="score-col"><span class="score {score_class}">{score}</span></td>
                <td>{result["reasoning"]}</td>
            </tr>
        """

    html += """
            </tbody>
        </table>
    </body>
    </html>
    """
    return html

#### `PromptEvaluator` class

Same class you saw in the prompt-evaluation module. The two methods each section will call:

- `generate_dataset()` — asks Claude to invent diverse athlete profiles + per-case solution criteria (we call this **once**, below).
- `run_evaluation()` — runs your `run_prompt_vN` against every test case, grades each output, prints the average score, and writes the HTML report.

In [18]:
class PromptEvaluator:
    def __init__(self, max_concurrent_tasks=3):
        self.max_concurrent_tasks = max_concurrent_tasks

    def render(self, template_string, variables):
        placeholders = re.findall(r"{([^{}]+)}", template_string)
        result = template_string
        for placeholder in placeholders:
            if placeholder in variables:
                result = result.replace("{" + placeholder + "}", str(variables[placeholder]))
        return result.replace("{{", "{").replace("}}", "}")

    def generate_unique_ideas(self, task_description, prompt_inputs_spec, num_cases):
        """Generate a list of unique ideas for test cases based on the task description"""
        prompt = """
        Generate {num_cases} unique, diverse ideas for testing a prompt that accomplishes this task:

        <task_description>
        {task_description}
        </task_description>

        The prompt will receive the following inputs
        <prompt_inputs>
        {prompt_inputs}
        </prompt_inputs>

        Each idea should represent a distinct scenario or example that tests different aspects of the task.

        Output Format:
        Provide your response as a structured JSON array where each item is a brief description of the idea.

        Example:
        ```json
        [
            "Testing with technical computer science terminology",
            "Testing with medical research findings",
            "Testing with complex mathematical concepts"
        ]
        ```

        Ensure each idea is:
        - Clearly distinct from the others
        - Relevant to the task description
        - Specific enough to guide generation of a full test case
        - Quick to solve without requiring extensive computation or multi-step processing
        - Solvable with no more than 400 tokens of output

        Remember, only generate {num_cases} unique ideas
        """

        system_prompt = "You are a test scenario designer specialized in creating diverse, unique testing scenarios."

        example_prompt_inputs = ""
        for key, value in prompt_inputs_spec.items():
            val = value.replace("\n", "\\n")
            example_prompt_inputs += f'"{key}": str # {val},'

        rendered_prompt = self.render(
            dedent(prompt),
            {
                "task_description": task_description,
                "num_cases": num_cases,
                "prompt_inputs": example_prompt_inputs,
            },
        )

        messages = []
        add_user_message(messages, rendered_prompt)
        add_assistant_message(messages, "```json")
        text = chat(messages, stop_sequences=["```"], system=system_prompt, temperature=1.0)
        return json.loads(text)

    def generate_test_case(self, task_description, idea, prompt_inputs_spec={}):
        """Generate a single test case based on the task description and a specific idea"""
        example_prompt_inputs = ""
        for key, value in prompt_inputs_spec.items():
            val = value.replace("\n", "\\n")
            example_prompt_inputs += f'"{key}": "EXAMPLE_VALUE", // {val}\n'

        allowed_keys = ", ".join([f'"{k}"' for k in prompt_inputs_spec.keys()])

        prompt = """
        Generate a single detailed test case for a prompt evaluation based on:

        <task_description>
        {task_description}
        </task_description>

        <specific_idea>
        {idea}
        </specific_idea>

        <allowed_input_keys>
        {allowed_keys}
        </allowed_input_keys>

        Output Format:
        ```json
        {{
            "prompt_inputs": {{
            {example_prompt_inputs}
            }},
            "solution_criteria": ["criterion 1", "criterion 2"]
        }}
        ```

        IMPORTANT REQUIREMENTS:
        - You MUST ONLY use these exact input keys in your prompt_inputs: {allowed_keys}
        - Do NOT add any additional keys to prompt_inputs
        - All keys listed in allowed_input_keys must be included in your response
        - Make the test case realistic and practically useful
        - Include measurable, concise solution criteria (1 to 4 items)
        - The solution criteria should ONLY address the direct requirements of the task description and the generated prompt_inputs
        - Avoid over-specifying criteria with requirements that go beyond the core task
        - Keep solution criteria simple, focused, and directly tied to the fundamental task
        - The test case should be tailored to the specific idea provided
        - Quick to solve without requiring extensive computation or multi-step processing
        - Solvable with no more than 400 tokens of output
        - DO NOT include any fields beyond those specified in the output format
        """

        system_prompt = "You are a test case creator specializing in designing evaluation scenarios."

        rendered_prompt = self.render(
            dedent(prompt),
            {
                "allowed_keys": allowed_keys,
                "task_description": task_description,
                "idea": idea,
                "example_prompt_inputs": example_prompt_inputs,
            },
        )

        messages = []
        add_user_message(messages, rendered_prompt)
        add_assistant_message(messages, "```json")
        text = chat(messages, stop_sequences=["```"], system=system_prompt, temperature=0.7)

        test_case = json.loads(text)
        test_case["task_description"] = task_description
        test_case["scenario"] = idea
        return test_case

    def generate_dataset(self, task_description, prompt_inputs_spec={}, num_cases=1, output_file="dataset.json"):
        """Generate test dataset based on task description and save to file"""
        ideas = self.generate_unique_ideas(task_description, prompt_inputs_spec, num_cases)
        dataset = []
        with concurrent.futures.ThreadPoolExecutor(max_workers=self.max_concurrent_tasks) as executor:
            future_to_idea = {
                executor.submit(self.generate_test_case, task_description, idea, prompt_inputs_spec): idea
                for idea in ideas
            }
            for future in concurrent.futures.as_completed(future_to_idea):
                try:
                    dataset.append(future.result())
                except Exception as e:
                    print(f"Error generating test case: {e}")

        with open(output_file, "w") as f:
            json.dump(dataset, f, indent=2)
        print(f"Generated {len(dataset)} test cases → {output_file}")
        return dataset

    def grade_output(self, test_case, output, extra_criteria):
        """Grade the output of a test case using the model"""
        prompt_inputs = ""
        for key, value in test_case["prompt_inputs"].items():
            val = str(value).replace("\n", "\\n")
            prompt_inputs += f'"{key}":"{val}",\n'

        extra_criteria_section = ""
        if extra_criteria:
            extra_criteria_template = """
            Mandatory Requirements - ANY VIOLATION MEANS AUTOMATIC FAILURE (score of 3 or lower):
            <extra_important_criteria>
            {extra_criteria}
            </extra_important_criteria>
            """
            extra_criteria_section = self.render(dedent(extra_criteria_template), {"extra_criteria": extra_criteria})

        eval_template = """
        Your task is to evaluate the following AI-generated solution with EXTREME RIGOR.

        Original task description:
        <task_description>
        {task_description}
        </task_description>

        Original task inputs:
        <task_inputs>
        {{ {prompt_inputs} }}
        </task_inputs>

        Solution to Evaluate:
        <solution>
        {output}
        </solution>

        Criteria you should use to evaluate the solution:
        <criteria>
        {solution_criteria}
        </criteria>

        {extra_criteria_section}

        Scoring Guidelines:
        * Score 1-3: Solution fails to meet one or more MANDATORY requirements
        * Score 4-6: Solution meets all mandatory requirements but has significant deficiencies in secondary criteria
        * Score 7-8: Solution meets all mandatory requirements and most secondary criteria, with minor issues
        * Score 9-10: Solution meets all mandatory and secondary criteria

        IMPORTANT SCORING INSTRUCTIONS:
        * Grade the output based ONLY on the listed criteria. Do not add your own extra requirements.
        * If a solution meets all of the mandatory and secondary criteria give it a 10
        * ANY violation of a mandatory requirement MUST result in a score of 3 or lower
        * The full 1-10 scale should be utilized - don't hesitate to give low scores when warranted

        Output Format
        Provide your evaluation as a structured JSON object with these fields in order:
        - "strengths": An array of 1-3 key strengths
        - "weaknesses": An array of 1-3 key areas for improvement
        - "reasoning": A concise explanation of your overall assessment
        - "score": A number between 1-10

        Respond with JSON. Keep your response concise and direct.
        Example response shape:
        {{
            "strengths": string[],
            "weaknesses": string[],
            "reasoning": string,
            "score": number
        }}
        """

        eval_prompt = self.render(
            dedent(eval_template),
            {
                "task_description": test_case["task_description"],
                "prompt_inputs": prompt_inputs,
                "output": output,
                "solution_criteria": "\n".join(test_case["solution_criteria"]),
                "extra_criteria_section": extra_criteria_section,
            },
        )

        messages = []
        add_user_message(messages, eval_prompt)
        add_assistant_message(messages, "```json")
        eval_text = chat(messages, stop_sequences=["```"], temperature=0.0)
        return json.loads(eval_text)

    def run_test_case(self, test_case, run_prompt_function, extra_criteria=None):
        """Run a test case and grade the result"""
        output = run_prompt_function(test_case["prompt_inputs"])
        model_grade = self.grade_output(test_case, output, extra_criteria)
        return {
            "output": output,
            "test_case": test_case,
            "score": model_grade["score"],
            "reasoning": model_grade["reasoning"],
        }

    def run_evaluation(self, run_prompt_function, dataset_file, extra_criteria=None,
                       json_output_file="output.json", html_output_file="output.html"):
        """Run evaluation on all test cases in the dataset"""
        with open(dataset_file, "r") as f:
            dataset = json.load(f)

        results = []
        with concurrent.futures.ThreadPoolExecutor(max_workers=self.max_concurrent_tasks) as executor:
            future_to_test_case = {
                executor.submit(self.run_test_case, tc, run_prompt_function, extra_criteria): tc
                for tc in dataset
            }
            for future in concurrent.futures.as_completed(future_to_test_case):
                results.append(future.result())

        average_score = mean([r["score"] for r in results])
        print(f"Average score: {average_score:.2f}")

        with open(json_output_file, "w") as f:
            json.dump(results, f, indent=2)

        html = generate_prompt_evaluation_report(results)
        with open(html_output_file, "w", encoding="utf-8") as f:
            f.write(html)

        return results

### Create the evaluator, generate the shared dataset, and set the extra criteria

Three things happen below and **they only happen once** — every section reuses them:

1. Create a `PromptEvaluator` instance with `max_concurrent_tasks=3`. Drop this to `1` if you hit rate limits.
2. Call `generate_dataset()` with `num_cases=3` to produce a handful of diverse athlete profiles. `num_cases=3` is the live-class default; bump to 10+ offline for tighter numbers.
3. Define `EXTRA_CRITERIA` — the hard requirements the grader will enforce on every version (caloric total, macros, meals with exact foods/portions/timing).

After this cell runs, every section below is just: *write a new `run_prompt_vN()`, call `evaluator.run_evaluation(...)`, read the score.*

`scores` is a small dict we'll populate as we go so section 6 can show the full climb side by side.

In [19]:
evaluator = PromptEvaluator(max_concurrent_tasks=3)

dataset = evaluator.generate_dataset(
    task_description="Write a compact, concise 1 day meal plan for a single athlete",
    prompt_inputs_spec={
        "height_cm": "Athlete's height in cm",
        "weight_kg": "Athlete's weight in kg",
        "goal": "Goal of the athlete",
        "dietary_restrictions": "Dietary restrictions of the athlete",
    },
    output_file="dataset.json",
    num_cases=3,
)

EXTRA_CRITERIA = """
The output should include:
- Daily caloric total
- Macronutrient breakdown
- Meals with exact foods, portions, and timing
"""

# Will accumulate {version_name: average_score} as we run each section.
scores = {}

Generated 3 test cases → dataset.json


---
# 1. Prompt Engineering — the loop

**Prompt engineering** is the practice of improving prompts to get more reliable, higher-quality outputs from language models. It's not "magic words" — it's a measurable, iterative loop.

### The loop (this is the whole module)

```
write initial prompt  →  interpolate inputs  →  run evaluation
         ↑                                              |
         └──  apply one technique  ←──────  read score ──┘
```

### Running example
Generate a **one-day meal plan for an athlete** using their height, weight, physical goal, and dietary restrictions.

### Baseline we're trying to beat
Starting with a naive prompt on Haiku, expect a score around **2.32 / 10**. We'll climb from there, one technique per section, and re-run the eval every time.

### Why this loop matters
Without an eval, "the prompt got better" is a feeling. With an eval, it's a number. Every technique in the rest of this notebook is something we *apply and immediately re-measure*.

### Demo: baseline prompt + first eval run

`run_prompt_v1_initial` is a deliberately weak prompt: no action verb, no structure, no format. Right after defining it, we call `evaluator.run_evaluation(...)` against the dataset we generated in setup. **Watch the printed average score** — that's our baseline. We'll come back to it five times.

This cell also writes `output_v1_initial.html`. Open it in a browser during class: every test case, the generated plan, and the grader's reasoning are all there.

In [20]:
def run_prompt_v1_initial(prompt_inputs):
    prompt = f"""Meal plan for an athlete.
Height: {prompt_inputs['height_cm']} cm
Weight: {prompt_inputs['weight_kg']} kg
Goal: {prompt_inputs['goal']}
Restrictions: {prompt_inputs['dietary_restrictions']}
"""
    messages = []
    add_user_message(messages, prompt)
    return chat(messages, temperature=0.0)


results = evaluator.run_evaluation(
    run_prompt_function=run_prompt_v1_initial,
    dataset_file="dataset.json",
    extra_criteria=EXTRA_CRITERIA,
    json_output_file="output_v1_initial.json",
    html_output_file="output_v1_initial.html",
)
scores["v1_initial"] = mean(r["score"] for r in results)

Average score: 8.00


> **🏫 During class:**
> 1. Run the cell and **write the score on a whiteboard / pin it in chat**. This is the baseline we'll compare every future run against.
> 2. Say out loud: *"Expected ~2.32. That's where a naive prompt lands on Haiku. By section 5 we're aiming for 8-plus on the same eval, same model, same athletes — only the prompt changes."*
> 3. Open `output_v1_initial.html` in a browser. Read one low-scoring test case and its grader reasoning out loud. Ask the room: *"What's the grader complaining about? Which of the next four techniques would fix it?"*

---
# 2. Being Clear and Direct

**Being clear and direct** = use simple, direct language with an **action verb** in the **first line** of the prompt to specify the exact task.

### Why the first line matters most
The first line sets the foundation for everything the model generates. If it's vague, every token afterwards is trying to guess what you actually wanted.

### Structure
```
<action verb> <direct task> <output specifications>
```

### Examples
| Vague | Clear and direct |
|---|---|
| *"solar panels"* | *"Write three paragraphs about how solar panels work."* |
| *"geothermal info"* | *"Identify three countries that use geothermal energy and for each include generation stats."* |
| *"meal plan for athlete"* | *"Generate a one-day meal plan for an athlete that meets their dietary restrictions."* |

### Expected lift
On the meal-plan eval, this single change took the score from **2.32 → 3.92**. No model change. No examples. Just a better first line.

### Demo: rewrite the first line, then re-run the eval

The only change from v1 is the first sentence — it now starts with an action verb (*Generate*) and states exactly what we want. The interpolated athlete data is identical. After defining `run_prompt_v2_clear_direct`, we immediately call `evaluator.run_evaluation(...)` on the **same dataset** to get an apples-to-apples score.

In [21]:
def run_prompt_v2_clear_direct(prompt_inputs):
    prompt = f"""Generate a one-day meal plan for an athlete that meets their dietary restrictions.

Height: {prompt_inputs['height_cm']} cm
Weight: {prompt_inputs['weight_kg']} kg
Goal: {prompt_inputs['goal']}
Restrictions: {prompt_inputs['dietary_restrictions']}
"""
    messages = []
    add_user_message(messages, prompt)
    return chat(messages, temperature=0.0)


results = evaluator.run_evaluation(
    run_prompt_function=run_prompt_v2_clear_direct,
    dataset_file="dataset.json",
    extra_criteria=EXTRA_CRITERIA,
    json_output_file="output_v2_clear_direct.json",
    html_output_file="output_v2_clear_direct.html",
)
scores["v2_clear_direct"] = mean(r["score"] for r in results)
print(f"  v1 baseline:    {scores['v1_initial']:.2f}")
print(f"  v2 clear+direct: {scores['v2_clear_direct']:.2f}")

Average score: 8.33
  v1 baseline:    8.00
  v2 clear+direct: 8.33


> **🏫 During class:**
> 1. Diff v1 and v2 on screen — **one line changed**. Everything else is identical.
> 2. Say out loud: *"That one-line change alone moves the eval from ~2.32 to ~3.92. This is the cheapest win in prompt engineering."*
> 3. Ask a student to propose an even stronger first line (e.g., specifying meals, calories, timing). Try it live, re-run the cell, and note that specificity is *section 3*.

---
# 3. Being Specific (attributes + steps)

**Being specific** = add **guidelines** that steer the output in a particular direction. There are two flavors, and they do different jobs.

| | **Type A — Attributes** | **Type B — Steps** |
|---|---|---|
| Controls | The **output's** qualities | The **model's reasoning process** |
| Example | "Include calorie totals. Format as a table. Use metric units." | "First compute TDEE. Then split macros. Then assign foods." |
| When to use | Almost **always** | Complex problems needing broader perspective |

### Combine them
Professional prompts almost always stack both: Type B guides *how* the model thinks; Type A constrains *what* it emits.

### Expected lift
On the meal-plan eval, adding guidelines jumped the score from **3.92 → 7.86**. This is usually the single biggest win in the loop.

### Demo: stack Type B (steps) + Type A (attributes), then re-run the eval

`run_prompt_v3_specific` keeps the clear first line from v2 and adds two blocks:
- **Reasoning steps** telling the model how to approach the task (compute TDEE → set macros → assign foods).
- **Output attributes** pinning the format (3 meals + 2 snacks, per-item grams, total calories + macros at the bottom).

Re-run the eval. This is the cell where the score usually makes its biggest jump.

In [22]:
def run_prompt_v3_specific(prompt_inputs):
    prompt = f"""Generate a one-day meal plan for an athlete that meets their dietary restrictions.

Follow these steps:
1. Estimate the athlete's daily calorie target using the Mifflin-St Jeor equation and an activity multiplier appropriate for their goal.
2. Split the target into macros (protein g, carbs g, fat g) suitable for endurance + lean-muscle goals.
3. Choose specific foods that hit those macros AND respect every dietary restriction.
4. Distribute the foods across breakfast, lunch, dinner, and two snacks.

The output must:
- Use a single Markdown table with columns: Meal | Item | Portion (g) | Calories | Protein (g).
- Include breakfast, lunch, dinner, and two snacks (5 rows minimum).
- End with a one-line total for calories and macros.
- Use metric units throughout.

Athlete:
- Height: {prompt_inputs['height_cm']} cm
- Weight: {prompt_inputs['weight_kg']} kg
- Goal: {prompt_inputs['goal']}
- Restrictions: {prompt_inputs['dietary_restrictions']}
"""
    messages = []
    add_user_message(messages, prompt)
    return chat(messages, temperature=0.0)


results = evaluator.run_evaluation(
    run_prompt_function=run_prompt_v3_specific,
    dataset_file="dataset.json",
    extra_criteria=EXTRA_CRITERIA,
    json_output_file="output_v3_specific.json",
    html_output_file="output_v3_specific.html",
)
scores["v3_specific"] = mean(r["score"] for r in results)
for name, s in scores.items():
    print(f"  {name:>18}  {s:.2f}")

Average score: 7.00
          v1_initial  8.00
     v2_clear_direct  8.33
         v3_specific  7.00


> **🏫 During class:**
> 1. Before running, predict the score out loud: *"I expect this jumps from ~3.9 to ~7.8."* Let the class hold you accountable when the number lands.
> 2. Open `output_v3_specific.html` next to `output_v2_clear_direct.html`. Find the same test case in both and read v2's output, then v3's output. The contrast makes the number real.
> 3. Variation to try live: delete just the **"Follow these steps"** block and re-run. The table usually still appears (Type A holds the format) but the macro math often gets sloppier — that's what Type B was buying us.

---
# 4. Structure with XML Tags

**XML tags** organize and delineate different content sections inside a prompt so the model can tell input apart from instructions, one document from another, and examples from the task.

### Why they help
When you interpolate large amounts of content into a prompt — a document, a user record, a stack of examples — plain text runs together. Tags give the model explicit boundaries.

### Pick descriptive names
| Weak | Strong |
|---|---|
| `<data>` | `<sales_records>` |
| `<text>` | `<athlete_information>` |
| `<code>` | `<my_code>` / `<docs>` |

A debugging prompt with code **and** documentation becomes clearer when the two are in `<my_code>` and `<docs>` blocks. Same content, obvious structure.

### When to wrap
- Any interpolated input, **even if it's short**. `<athlete_information>...</athlete_information>` is worth doing for four lines of data.
- Any time you have more than one "thing" in a prompt — input + instructions, task + examples, two documents, etc.

### Demo: wrap the athlete data in a descriptive tag, then re-run the eval

`run_prompt_v4_xml` is the same as v3 with one surgical change: the four athlete fields are now inside `<athlete_information>...</athlete_information>`. **Watch:** on four lines of input the score may barely budge — XML tags earn their keep on **longer, messier inputs** (medical intake forms, logs, multi-doc retrieval). The habit is worth forming on short inputs so it's automatic on long ones.

In [23]:
def run_prompt_v4_xml(prompt_inputs):
    prompt = f"""Generate a one-day meal plan for an athlete that meets their dietary restrictions.

Follow these steps:
1. Estimate the athlete's daily calorie target using the Mifflin-St Jeor equation and an activity multiplier appropriate for their goal.
2. Split the target into macros (protein g, carbs g, fat g) suitable for endurance + lean-muscle goals.
3. Choose specific foods that hit those macros AND respect every dietary restriction.
4. Distribute the foods across breakfast, lunch, dinner, and two snacks.

The output must:
- Use a single Markdown table with columns: Meal | Item | Portion (g) | Calories | Protein (g).
- Include breakfast, lunch, dinner, and two snacks (5 rows minimum).
- End with a one-line total for calories and macros.
- Use metric units throughout.

<athlete_information>
Height: {prompt_inputs['height_cm']} cm
Weight: {prompt_inputs['weight_kg']} kg
Goal: {prompt_inputs['goal']}
Restrictions: {prompt_inputs['dietary_restrictions']}
</athlete_information>
"""
    messages = []
    add_user_message(messages, prompt)
    return chat(messages, temperature=0.0)


results = evaluator.run_evaluation(
    run_prompt_function=run_prompt_v4_xml,
    dataset_file="dataset.json",
    extra_criteria=EXTRA_CRITERIA,
    json_output_file="output_v4_xml.json",
    html_output_file="output_v4_xml.html",
)
scores["v4_xml"] = mean(r["score"] for r in results)
for name, s in scores.items():
    print(f"  {name:>18}  {s:.2f}")

Average score: 8.00
          v1_initial  8.00
     v2_clear_direct  8.33
         v3_specific  7.00
              v4_xml  8.00


> **🏫 During class:**
> 1. Show the diff from v3 — the only change is four lines wrapped in a tag. Say: *"You do this even for four lines. Habit now, payoff later."*
> 2. If the score barely moves, use that: *"On four-line inputs the model doesn't need the boundary. Now imagine pasting a whole patient chart in — that's when this pays off."*
> 3. Variation to run live: rename the tag to a useless one like `<data>` and re-run. Often the model's referencing gets vaguer (*"based on the data provided…"*). Then rename to `<athlete_information>` — the response starts referencing the athlete by attribute.

---
# 5. Providing Examples (one-shot / multi-shot)

**One-shot / multi-shot prompting** = include one (or several) worked examples in the prompt so the model can pattern-match on both the format and the quality bar.

### When it helps most
| Use case | Why examples help |
|---|---|
| Corner cases (sarcasm, edge inputs) | Instructions can't enumerate every case; examples show the judgment call. |
| Complex output formats (JSON, nested structures) | An example is worth a thousand "make sure to include…" bullets. |
| Specific style / tone | Copying style from prose is easier than defining it. |

### How to structure an example
- Wrap it in XML tags so it's obviously a reference — not the current task.
- Include **both** the sample input and the ideal output.
- Add a short note explaining **why** the output is ideal — this reinforces the qualities you want copied.
- Add explicit guidance for corner cases (*"be especially careful with sarcasm"*, *"never assume the athlete eats eggs unless stated"*).

### Where to put examples
**After** the instructions and output spec, **before** the actual task input. Order: instructions → examples → task.

### Pro tip
Pull your highest-scoring outputs from the prompt-evaluation module and reuse them here as examples. The eval has already told you which outputs are best — that's your example library.

### Demo: one-shot example with reasoning, then re-run the eval

`run_prompt_v5_one_shot` layers all four prior techniques and adds a single worked example: a different athlete profile with an ideal meal plan *and* a `<why_this_is_ideal>` block explaining what makes that plan good. Re-run the eval — this is the last prompt version; the score here is where we landed.

In [24]:
def run_prompt_v5_one_shot(prompt_inputs):
    prompt = f"""Generate a one-day meal plan for an athlete that meets their dietary restrictions.

Follow these steps:
1. Estimate the athlete's daily calorie target using the Mifflin-St Jeor equation and an activity multiplier appropriate for their goal.
2. Split the target into macros (protein g, carbs g, fat g) suitable for endurance + lean-muscle goals.
3. Choose specific foods that hit those macros AND respect every dietary restriction.
4. Distribute the foods across breakfast, lunch, dinner, and two snacks.

The output must:
- Use a single Markdown table with columns: Meal | Item | Portion (g) | Calories | Protein (g).
- Include breakfast, lunch, dinner, and two snacks (5 rows minimum).
- End with a one-line total for calories and macros.
- Use metric units throughout.
- Respect every dietary restriction exactly — do not substitute “close” foods.

<example>
<athlete_information>
Height: 165 cm
Weight: 58 kg
Goal: Olympic-distance triathlon prep
Restrictions: gluten-free, no shellfish
</athlete_information>
<ideal_output>
**Daily target: 2450 kcal — 130 g protein / 320 g carbs / 70 g fat**

| Meal      | Item                          | Portion (g) | Calories | Protein (g) |
|-----------|-------------------------------|-------------|----------|-------------|
| Breakfast | Rolled oats (GF) + banana     | 80 + 120    | 430      | 11          |
| Snack 1   | Greek yogurt + honey          | 200 + 20    | 220      | 20          |
| Lunch     | Grilled chicken + rice + veg  | 150 + 200 + 150 | 680  | 42          |
| Snack 2   | Rice cakes + almond butter    | 40 + 30     | 320      | 9           |
| Dinner    | Salmon + quinoa + greens      | 170 + 180 + 150 | 800  | 48          |

**Total: 2450 kcal — 130 g protein / 318 g carbs / 72 g fat**
</ideal_output>
<why_this_is_ideal>
- Hits the calorie/macro target within 5 kcal and 5 g of each macro.
- Every item is naturally gluten-free and contains no shellfish.
- Meals scale around heavy-training days.
- Markdown table matches the required column order exactly.
</why_this_is_ideal>
</example>

<athlete_information>
Height: {prompt_inputs['height_cm']} cm
Weight: {prompt_inputs['weight_kg']} kg
Goal: {prompt_inputs['goal']}
Restrictions: {prompt_inputs['dietary_restrictions']}
</athlete_information>
"""
    messages = []
    add_user_message(messages, prompt)
    return chat(messages, temperature=0.0)


results = evaluator.run_evaluation(
    run_prompt_function=run_prompt_v5_one_shot,
    dataset_file="dataset.json",
    extra_criteria=EXTRA_CRITERIA,
    json_output_file="output_v5_one_shot.json",
    html_output_file="output_v5_one_shot.html",
)
scores["v5_one_shot"] = mean(r["score"] for r in results)

print("\n=== Full score climb ===")
for name, s in scores.items():
    bar = "#" * int(round(s))
    print(f"  {name:>18}  {s:5.2f}  {bar}")

Average score: 7.67

=== Full score climb ===
          v1_initial   8.00  ########
     v2_clear_direct   8.33  ########
         v3_specific   7.00  #######
              v4_xml   8.00  ########
         v5_one_shot   7.67  ########


> **🏫 During class:**
> 1. Before running, call out the structure on screen: *instructions first, then `<example>` with input AND ideal output AND `<why_this_is_ideal>`, then the real `<athlete_information>`*. Order matters.
> 2. Run the cell. Read the final **Full score climb** table out loud. Circle the biggest jump (usually between v2 and v3) and the smallest (often v3 → v4). That shape is the lesson.
> 3. Variation to try live: delete the `<why_this_is_ideal>` block and re-run. The format usually still copies, but the quality signals drop — that's what the reasoning was buying us.
> 4. Ask: *"Where do we get good examples from?"* — answer: the high-scoring outputs from last week's eval. Open `output_v5_one_shot.html`, find the 10/10 case, and that's your next example.

---
# 6. Recap + practice exercises

### Recap (run through these out loud)
- **Prompt engineering is a loop:** write → eval → read score → apply one technique → re-run. Don't eyeball improvements; measure them.
- **Be clear and direct:** action verb in the first line. Cheapest single win (2.32 → 3.92 on our eval).
- **Be specific:** attributes (Type A) constrain the output; steps (Type B) guide the reasoning. Combine both (3.92 → 7.86).
- **XML tags:** give interpolated content explicit boundaries and descriptive names — even for short content. On tiny inputs the score may barely move; the win shows up on long, messy inputs.
- **Examples:** one-shot or multi-shot with a `<why_this_is_ideal>` block transfers both format and quality bar. Harvest them from your highest-scoring eval runs.
- **Order inside a prompt:** instructions → output spec → examples → task input.
- **The eval is the ground truth.** Every section here ran the same `evaluator.run_evaluation(...)` against the same dataset — the only variable was the prompt body.

### Exercises (do the first in class, assign the rest)
1. **Rewrite the first line:** Take your own starting prompt from the eval module, rewrite only the first line using an action verb, wrap it in a new `run_prompt_exercise1(prompt_inputs)`, and run `evaluator.run_evaluation(...)`. Record the score change.
2. **Stack attributes + steps:** Add a *“Follow these steps…”* block AND an *“The output must…”* block to that prompt. Re-run the eval.
3. **Wrap your inputs:** Replace every interpolated field with a descriptive XML tag. Re-run.
4. **One-shot from your eval:** Pick your highest-scoring output from exercise 2 (open the `output_*.html` to find it), convert it into an `<example>` block with `<ideal_output>` and `<why_this_is_ideal>`, and add it to the prompt. Re-run.
5. **Multi-shot for a corner case:** Add a second example that covers a deliberately tricky profile (e.g., severe allergies + unusual goal). Verify the score on tricky test cases specifically improves more than the score on easy cases — inspect `output_*.html` to confirm.

In [13]:
# Exercise 1 scaffold — finish this live in class together.
# Uncomment to run.
#
# def run_prompt_exercise1(prompt_inputs):
#     # Start from a weak prompt and rewrite ONLY the first line using an action verb.
#     prompt = f"""meal plan for athlete
# Height: {prompt_inputs['height_cm']} cm
# Weight: {prompt_inputs['weight_kg']} kg
# Goal: {prompt_inputs['goal']}
# Restrictions: {prompt_inputs['dietary_restrictions']}
# """
#     # TODO: rewrite the first line using an action verb + direct task + output specs.
#     messages = []
#     add_user_message(messages, prompt)
#     return chat(messages, temperature=0.0)
#
# results = evaluator.run_evaluation(
#     run_prompt_function=run_prompt_exercise1,
#     dataset_file="dataset.json",
#     extra_criteria=EXTRA_CRITERIA,
#     json_output_file="output_exercise1.json",
#     html_output_file="output_exercise1.html",
# )
# scores["exercise1"] = mean(r["score"] for r in results)
# for name, s in scores.items():
#     print(f"  {name:>18}  {s:.2f}")